# Loan approval modelling — modular pipeline

This version keeps EDA optional and places all model-time transformations inside a scikit-learn `Pipeline`. That means feature engineering, imputation, encoding, and scaling are learned from the training folds only, preventing data leakage during validation and hyperparameter tuning.

Update `DATA_PATH` if your CSV is stored elsewhere. `RiskScore` is excluded by default because it may be a derived risk variable and could leak the approval decision; keep it only after confirming it is available at application time and is not calculated from the approval outcome.


## 1. Imports and configuration


In [ ]:
from pathlib import Path
import pickle
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore') # ignoring warnings

RANDOM_STATE = 42
TARGET = 'LoanApproved'
DATA_PATH = Path('./data files/Loan.csv')
MODEL_DIR = Path('./trained models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Remove columns that are unavailable at decision time or would leak the target.
LEAKAGE_OR_UNUSED_COLUMNS = ['RiskScore']


## 2. Data loading and optional EDA helpers


In [ ]:
# Instead of treating file paths as error prone string, we wrap them inside a Path object which allows object oriented file system path handling.
def load_data(path: Path) -> pd.DataFrame:
    """Load the raw loan dataset and validate the target column."""
    data = pd.read_csv(path)
    if TARGET not in data.columns:
        raise ValueError(f"Expected target column '{TARGET}' was not found.")
    return data


def dataset_overview(data: pd.DataFrame) -> pd.DataFrame:
    """Return a compact schema and missing-value summary."""
    return pd.DataFrame({
        'dtype': data.dtypes.astype(str),
        'missing_values': data.isna().sum(),
        'unique_values': data.nunique(),
    }).sort_index()


# def plot_target_distribution(data: pd.DataFrame, target: str = TARGET) -> None:
#     """Plot the approval-class balance."""
#     ax = sns.countplot(data=data, x=target, palette='Set2')
#     ax.set_title('Loan approval distribution')
#     ax.set_xlabel('Loan approved (0 = denied, 1 = approved)')
#     plt.show()



In [ ]:
df = load_data(DATA_PATH)
print(f'Dataset shape: {df.shape}')
display(dataset_overview(df))
# plot_target_distribution(df)

## 3. Feature engineering

The transformer below receives raw data. It creates ratios, corrects/extracts the application year and month, applies `log1p` to non-negative financial variables, and drops columns that should not enter the classifier. Because it is part of the pipeline, every cross-validation fold learns downstream preprocessing only from its own training data.


This is to give a little context about the class that we are implementing below.    
So, DataFeatureEngineering inherits from 2 classes - BaseEstimator and TransformerMixin.

#### 1) BaseEstimator
- BaseEstimator is a helper class in scikit-learn. When your custom class inherits from it, you automatically get two hidden methods: `get_params()` and `set_params()`.
- Why do you need get_params and set_params?
    - These two methods allow scikit-learn's automated tools (like GridSearchCV or RandomizedSearchCV) to "see" inside your class. Consider the below code:
    ```{python}
            # 1. Define your custom class with BaseEstimator
            class CustomImputer(BaseEstimator, TransformerMixin):
                def __init__(self, strategy='mean'):  # <--- 'strategy' is a hyperparameter
                    self.strategy = strategy
                    
                def fit(self, X, y=None):
                    return self
                    
                def transform(self, X):
                    # (Your code here to fill missing data using self.strategy)
                    return X

            # 2. Put it in a Pipeline
            pipeline = Pipeline([
                ('imputer', CustomImputer()),
                ('model', LogisticRegression())
            ])

            # 3. BaseEstimator allows GridSearchCV to automatically change 'strategy'!
            param_grid = {
                'imputer__strategy': ['mean', 'median', 'most_frequent']
            }

            grid_search = GridSearchCV(pipeline, param_grid)
            grid_search.fit(X_train, y_train) 
    ```

    Now, what the above code does is that it implements a GridSearchCV method to test out different methods of data imputation to check which one gives the best results. Let's break down the code to understand how it does that:
    - The param_grid contains the list of hyperparameters along with the values that you want to test
    - 'imputer__strategy' tells GridSearchCV to look into the `imputer` step inside the pipeline, __ means to look inside the `imputer` step. And inside the imputer step, look for the `strategy` hyperparameter.
    - This way, the naming convention of the param_grid tell the GridSearchCV which hyperparameters to look at and under which step of the pipeline.

    Now, say we want to also try out different values for the `max_iter` hyperparameter of the LogisticRegression model. Here is how the param_grid will change:
    ```{python}
        param_grid = {
            'imputer__strategy': ['mean', 'median', 'most_frequent'],
            'model__max_iter' : [100, 500, 1000]
        }
    ```


#### 2) TransformerMixin
- This creates a fit_transform() method for the class.
- You need to manually define a fit() and transform() method for the custom class that you create.
- In other words, it gives the capability of fit and transform to the Customer Class that we create (to fit and transform preprocessing stages like scaling and encoding to prepare our data to feed to the models).
- So when we run our pipeline, it automatically sets fit_transform() for our X_train data and transform() for our X_val and X_test data.

In [ ]:
class DataFeatureEngineering(BaseEstimator, TransformerMixin):
    """Create model features from raw loan application records.

    The class deliberately returns a DataFrame so later preprocessing can select
    columns by name. It does not estimate statistics from the full dataset.
    """
    # column that need to undergo log transformation
    log_transform_columns = [
        'SavingsAccountBalance', 'TotalLiabilities', 'TotalAssets',
        'CheckingAccountBalance', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio',
        'AnnualIncome', 'MonthlyIncome', 'LoanAmount', 'MonthlyDebtPayments', 'NetWorth'
    ]

    # columns to remove from the dataset
    columns_to_remove = [
        'ApplicationDate', 'AnnualIncome', 'Experience', 'TotalAssets',
        'TotalLiabilities', 'Monthly_LoanToIncomeRatio', 'MonthlyDebtPayments',
        'LoanAmount', 'MonthlyIncome', 'SavingsAccountBalance',
        'CheckingAccountBalance', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio',
        'NetWorth'
    ]

    def __init__(self, drop_columns=None):
        self.drop_columns = drop_columns

    def fit(self, X, y=None):
        return self

    # Static method: A function inside a class that does not need access to the class or instance itself 
    # (notice it doesn't take `self` as the first argument).
    # Regular methods like transform(self, X) need self to read/modify instance properties. A @staticmethod behaves like a 
    # normal standalone helper function, but is kept inside the class simply because it belongs to that class's namespace conceptually.
    @staticmethod
    def _correct_application_year(year: pd.Series) -> pd.Series:
        """Map anomalous encoded years to the intended 2018–2024 range."""
        corrected_years = year.copy()
        mapping = [
            (2025, 2031, 2018), (2032, 2038, 2019), (2039, 2045, 2020),
            (2046, 2052, 2021), (2053, 2059, 2022), (2060, 2066, 2023),
        ]
        for lower, upper, replacement in mapping:
            corrected_years = corrected_years.mask(corrected_years.between(lower, upper), replacement)
        return corrected_years.mask(corrected_years >= 2067, 2024)


    def transform(self, X):
        """Implementing feature engineering and applying log transformations"""

        data = X.copy()

        # Computing DebtToIncomeRatio and Monthly_LoanToIncomeRatio. We are making sure to avoid division by 0 error.
        if {'MonthlyDebtPayments', 'MonthlyIncome'}.issubset(data.columns):
            denominator = data['MonthlyIncome'].replace(0, np.nan)
            data['DebtToIncomeRatio'] = data['MonthlyDebtPayments'] / denominator

        if {'MonthlyLoanPayment', 'MonthlyIncome'}.issubset(data.columns):
            denominator = data['MonthlyIncome'].replace(0, np.nan)
            data['Monthly_LoanToIncomeRatio'] = data['MonthlyLoanPayment'] / denominator

        # extracting application year and month from the application date. 
        # correcting the application year and converting the application month to sin and cos values 
        # (to represent cyclic nature instead of inducing order in the features, that is, it tells the model that 
        # month 12 and month 1 are adjacent without implying that December is 12x "bigger" than January.)
        if 'ApplicationDate' in data.columns:
            dates = pd.to_datetime(data['ApplicationDate'], errors='coerce')
            data['ApplicationYear'] = self._correct_application_year(dates.dt.year)
            month = dates.dt.month
            data['Month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
            data['Month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)

        for column in self.log_transform_columns:
            if column in data.columns:
                # log1p to avoid taking a log of zero, which results in an infinite value. 
                # we are using np.lop1p instead of np.log to retain the extremely small precisions (to avoid being rounded to 0)
                # .clip(lower=0) -> replaces any negative numbers with 0
                data[f'{column}_log'] = np.log1p(data[column].clip(lower=0)) 

        # creating a set of columns that need to be removed from the final dataset 
        # (original columns before log transformation + any additional columns we want to drop (that you can specify later))
        removal_list = set(self.columns_to_remove) | set(self.drop_columns or [])

        # ** set(self.drop_columns or []) -> if drop_columns is None, it will choose the empty list, to ensure the Python code does not crash
        
        data.drop(columns=list(removal_list), errors='ignore', inplace = True)

        return data


## 4. Pipeline construction and data split


In [ ]:
def split_data(data: pd.DataFrame, target: str = TARGET, test_split: float = 0.5, validation_split: float = 0.3, random_state: int = RANDOM_STATE):
    """Create stratified train/validation/test splits from raw records."""

    X = data.drop(columns=[target])
    y = data[target]

    X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=validation_split, stratify=y, random_state=random_state)
    X_val, X_test, y_val, y_test = train_test_split(X_holdout, y_holdout, test_split=test_split, stratify=y_holdout, random_state=random_state)

    return X_train, X_val, X_test, y_train, y_val, y_test


def get_feature_groups(X_train: pd.DataFrame, drop_columns=None):
    """Infer column types after feature engineering without fitting scalers on all data."""
    feat_eng = DataFeatureEngineering(drop_columns=drop_columns)
    X_train_engineered = feat_eng.fit_transform(X_train) # applying feature engineering to train data 

    # Getting categorical and numeric features (to apply the respective encoding and scaling transformations)
    categorical = X_train_engineered.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    numeric = X_train_engineered.select_dtypes(include=np.number).columns.tolist()
    return numeric, categorical


def encoding_and_scaling(X_train: pd.DataFrame) -> pd.DataFrame:
    numeric_features, categorical_features = get_feature_groups(X_train, drop_columns=LEAKAGE_OR_UNUSED_COLUMNS)

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy = "most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown = "ignore"))
        # handle_unknown='ignore' is designed for production safety. 
        # If a new, unseen category appears in the validation or test set, OneHotEncoder simply sets all one-hot encoded 
        # columns for that row to 0.
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ], remainder='drop')

    return preprocessor


def build_pipeline(model, X_train: pd.DataFrame) -> Pipeline:
    """Build a single leakage-safe feature engineering + preprocessing + model pipeline."""

    feature_engineering = DataFeatureEngineering(drop_columns=LEAKAGE_OR_UNUSED_COLUMNS)
    preprocessor = encoding_and_scaling(X_train)

    return Pipeline([
        ('feature_engineering', feature_engineering),
        ('preprocessing', preprocessor),
        ('model', model),
    ])

Creating a pipeline has a rule: The last step must be a model (estimator), and all previous steps must be transformers.

So, when we call `pipeline.fit(X_train, y_train)`, the pipeline automatically first calls `.fit_transform()` (all the preprocessing steps) on the data. It then takes the final cleaned data and calls the `.fit(X_clean, y_train)`

## 5. Training and evaluation helpers


In [ ]:
# this function contains the code for the predictions on the evaluation metrics
# it then returns the metric scores
def evaluate_classifier(model: Pipeline, X: pd.DataFrame, y: pd.Series, dataset_name: str = 'Validation') -> dict:
    """Print and return classification metrics for a fitted pipeline."""
    y_pred = model.predict(X)
    y_probab = model.predict_proba(X)[:, 1]
    
    metrics = {
        'roc_auc': roc_auc_score(y, y_probab),
        'average_precision': average_precision_score(y, y_probab),
    }

    print(f'\n{dataset_name} classification report')
    print(classification_report(y, y_pred, digits=3))
    print(f"ROC-AUC: {metrics['roc_auc']:.3f} | Average precision: {metrics['average_precision']:.3f}")
    ConfusionMatrixDisplay.from_predictions(y, y_pred, cmap='Blues')
    plt.title(f'{dataset_name} confusion matrix')
    plt.show()
    
    return metrics

# this function builds a pipeline using the model and X_train
# it then trains the model pipeline 
# it then evaluates the trained pipeline using the evaluation metrics on the validation data
# then returns the trained model and the metric scores
def fit_and_evaluate(model, X_train, y_train, X_val, y_val):
    """Fit one pipeline and evaluate it on the validation split."""
    pipeline = build_pipeline(model, X_train)
    pipeline.fit(X_train, y_train)
    metrics = evaluate_classifier(pipeline, X_val, y_val)
    return pipeline, metrics

# this function compares the ROC-AUC and PR-AUC curves of ALL the evaluated models
def compare_roc_pr_curves(fitted_models: dict, X: pd.DataFrame, y: pd.Series) -> None:
    """Plot ROC and precision-recall curves for named fitted pipelines."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for name, model in fitted_models.items():
        probabilities = model.predict_proba(X)[:, 1]
        RocCurveDisplay.from_predictions(y, probabilities, name=name, ax=axes[0])
        PrecisionRecallDisplay.from_predictions(y, probabilities, name=name, ax=axes[1])
    axes[0].set_title('ROC curves')
    axes[1].set_title('Precision–recall curves')
    plt.tight_layout()
    plt.show()


def save_pipeline(model: Pipeline, filename: str) -> Path:
    """Saves the complete fitted pipeline, not just the estimator."""
    output_path = MODEL_DIR / filename
    with open(output_path, 'wb') as file:
        # pickle.dump(model, file)
        joblib.dump(model, file)
    return output_path


## 6. Baseline models

Each candidate uses the same preprocessing pipeline. This makes the comparison fair and guarantees that a future prediction receives identical transformations.


In [ ]:
# Splitting the raw data
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df,'LoanApproved')

In [ ]:
import xgboost as xgb

# {'model name' : model}
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    'XG Boost' : xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss',
            random_state=RANDOM_STATE, n_jobs=-1)
}

fitted_models = {}
validation_results = {}

for name, estimator in models.items():
    print(f'\n--- {name} ---')
    fitted_models[name], validation_results[name] = fit_and_evaluate(estimator, X_train, y_train, X_val, y_val)

results = pd.DataFrame(validation_results).T.sort_values('average_precision', ascending=False)
print(results)


## 7. Hyperparameter tuning

Notice the `model__` prefix: it tells the search to tune the model step within the pipeline. Cross-validation will therefore refit the complete preprocessing workflow separately in every fold.


In [ ]:
def tune_model(pipeline: Pipeline, parameter_space: dict, X_train, y_train, search_type: str = 'random', n_iter: int = 20, scoring: str = 'f1'):
    """Tune a full pipeline using cross-validation on the training set only."""
    if search_type == 'grid':
        search = GridSearchCV(pipeline, parameter_space, cv=5, scoring=scoring, n_jobs=-1)
    else:
        search = RandomizedSearchCV(pipeline, parameter_space, n_iter=n_iter, cv=5, scoring=scoring,random_state=RANDOM_STATE, n_jobs=-1)

    search.fit(X_train, y_train)
    print('Best parameters:', search.best_params_)
    print(f'Best CV {scoring}: {search.best_score_:.3f}')
    return search


rf_pipeline = build_pipeline(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1), 
    X_train
)
rf_parameter_space = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2'],
}


rf_search = tune_model(rf_pipeline, rf_parameter_space, X_train, y_train, n_iter=15)
best_model = rf_search.best_estimator_
evaluate_classifier(best_model, X_val, y_val, 'Validation (tuned Random Forest)')

In [ ]:
xgb_pipeline = build_pipeline(
    xgb.XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    X_train
)
xgb_parameter_space = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [3, 5, 7],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
}
xgb_search = tune_model(xgb_pipeline, xgb_parameter_space, X_train, y_train, n_iter=15) # runs random search by default
fitted_models['XGBoost'] = xgb_search.best_estimator_


## 8. Final test evaluation and saving

Choose the model based on validation/CV performance, retrain it on the combined training and validation data, and evaluate the test set exactly once.


In [ ]:
# Example: select the best fitted model after reviewing validation_results.
selected_name = max(validation_results, key=lambda name: validation_results[name]['average_precision'])
selected_model = fitted_models[selected_name]
print(f'Selected model: {selected_name}')

# Refit a fresh copy of the selected pipeline on train + validation before the final test.
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])
final_pipeline = build_pipeline(selected_model.named_steps['model'], X_train_final)
final_pipeline.fit(X_train_final, y_train_final)
test_metrics = evaluate_classifier(final_pipeline, X_test, y_test, 'Final test')
print(test_metrics)
saved_path = save_pipeline(final_pipeline, 'loan_approval_pipeline.pkl')
print(f'Complete pipeline saved to: {saved_path}')
